# Training a first model on Colab

Takes the patches `aral_colab.ipynb` left on Drive, labels them
automatically with the index thresholds, and trains two models on them:
a U-Net, then Prithvi-EO-2.0-300M-TL.

**Read this before trusting any number below.** There are no hand labels
yet, so the labels here are the threshold's opinion. A model trained on
them learns to *reproduce the threshold*, mistakes included. What this run
can show: the data loads, the models train, the loss falls, the masks look
sane. What it cannot show: whether a model beats the threshold. That
needs hand labels, and then only `labels.weak_labels` changes.

**Runtime > Change runtime type > T4 GPU.** On CPU this is unusably slow.

## 1. Libraries

`terratorch` may replace Colab's PyTorch with a newer build. If Colab then
offers *Restart session*, accept, and run this cell again - the second time
it installs nothing.

In [ ]:
USE_PRITHVI = True     # False skips Prithvi and its large install
INSTALL = True

import subprocess, sys
if INSTALL:
    pkgs = ['segmentation-models-pytorch', 'rasterio']
    if USE_PRITHVI:
        pkgs.append('terratorch')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *pkgs], check=True)

import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('torch', torch.__version__, '| device:',
      torch.cuda.get_device_name(0) if device == 'cuda' else
      'CPU - switch the runtime to a T4 GPU before training')

## 2. The code

Same cell as in `aral_colab.ipynb`: refreshes `src/` and re-imports it.

In [ ]:
REPO_URL = 'https://github.com/xdido2/aral-geo-map.git'  # e.g. 'https://github.com/<you>/aral-geo-map.git'

import os, sys, pathlib, subprocess
from google.colab import drive
drive.mount('/content/drive')

DRIVE = pathlib.Path('/content/drive/MyDrive/aral')
DRIVE.mkdir(parents=True, exist_ok=True)
ROOT = pathlib.Path('/content/aral-geo-map')


def _token():
    """GitHub token from Colab Secrets, or None. Never reaches the output."""
    try:
        from google.colab import userdata
        return userdata.get('GITHUB_TOKEN')
    except Exception:
        return None


def _run(args, tok):
    r = subprocess.run(args, capture_output=True, text=True)
    if r.returncode:
        msg = (r.stderr or r.stdout).strip()
        print('git failed:\n ', msg.replace(tok, '***') if tok else msg)
        if not tok:
            print('  (a private repo looks like a missing one here -')
            print('   add GITHUB_TOKEN to Colab Secrets if it is private)')
    return r.returncode == 0


def from_git(url):
    """Clone, or bring an existing clone up to date. Token stays off disk."""
    tok = _token()
    auth = url.replace('https://', f'https://{tok}@') if tok else url
    if ROOT.exists() and any(ROOT.iterdir()) and not (ROOT / '.git').exists():
        # Unpacked from the zip earlier, and perhaps holding downloaded
        # data: make it a repository in place instead of cloning over it.
        _run(['git', 'init', '--quiet', str(ROOT)], tok)
        _run(['git', '-C', str(ROOT), 'remote', 'add', 'origin', url], tok)
    if (ROOT / '.git').exists():
        # fetch by explicit URL, so no credential is ever stored in the repo
        if not _run(['git', '-C', str(ROOT), 'fetch', '--quiet', auth, 'main'], tok):
            return None
        # reset touches tracked files only; data/ is untracked and untouched
        if not _run(['git', '-C', str(ROOT), 'reset', '--hard', '--quiet',
                     'FETCH_HEAD'], tok):
            return None
        return f'{url} (updated)'
    if not _run(['git', 'clone', '--quiet', auth, str(ROOT)], tok):
        return None
    _run(['git', '-C', str(ROOT), 'remote', 'set-url', 'origin', url], tok)
    return f'{url} (cloned)'


def from_zip():
    src_zip = DRIVE / 'aral_src.zip'
    if not src_zip.exists():
        raise SystemExit(f'Upload dist/aral_src.zip to {DRIVE}, or set REPO_URL.')
    ROOT.mkdir(parents=True, exist_ok=True)
    # -o overwrites the code and leaves everything else alone
    subprocess.run(['unzip', '-qo', str(src_zip), '-d', str(ROOT)], check=True)
    return f'{src_zip.name} (unpacked)'


source = (from_git(REPO_URL) if REPO_URL else None) or from_zip()

os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
# Updating the files is not enough: a module already imported stays in
# memory, so a fix on disk would look like no fix at all.
for name in [m for m in list(sys.modules) if m.startswith('src.aral')]:
    del sys.modules[name]
import src.aral
from src.aral import config as C, labels

NOTEBOOK_REV = 2      # must equal src/aral/__init__.py
if getattr(src.aral, 'NOTEBOOK_REV', 0) != NOTEBOOK_REV:
    print('!' * 72)
    print('This page is out of step with the code it just pulled.')
    print('Reload the page (Cmd+R / Ctrl+R). The runtime and data/ survive it;')
    print('File > Open notebook would start a new runtime and lose them.')
    print('!' * 72)

print('source :', source)
print('classes:', C.CLASS_NAMES)
print('season :', C.SEASON, '| cells:', len(C.PATHS) * len(C.ROWS))

## 3. Settings

In [ ]:
PATCH_DIR = DRIVE / 'patches'      # written by aral_colab.ipynb
OUT_DIR = DRIVE / 'models'
MAX_PATCHES = None                 # e.g. 80 for a quick first try
EPOCHS_UNET = 15
EPOCHS_PRITHVI = 8
BATCH_UNET = 16
BATCH_PRITHVI = 8                  # ~6-7 GB on a T4 at one date per patch
SAVE_WEIGHTS = False               # weak-label weights only prove the loop;
                                   # Prithvi's take ~650 MB of Drive per run
SEED = 0

import numpy as np, random
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
OUT_DIR.mkdir(parents=True, exist_ok=True)

## 4. Patches and their automatic labels

Each patch is 256 x 256 pixels, six bands of reflectance. The label is one
class per pixel; `255` marks pixels with no data, which the loss ignores.

In [ ]:
import re
files = sorted(PATCH_DIR.glob('*.tif'))[:MAX_PATCHES]
assert files, f'no patches in {PATCH_DIR} - run aral_colab.ipynb first'

X, Y, places = [], [], []
for f in files:
    data, valid = labels.read_patch(f)
    X.append(labels.normalise(data, valid))
    Y.append(labels.weak_labels(data, valid))
    m = re.search(r'_x(\d+)y(\d+)\.tif$', f.name)
    places.append((int(m[1]), int(m[2])))
X, Y = np.stack(X), np.stack(Y)
print(f'{len(files)} patches, input {X.shape}, {X.nbytes/1e6:.0f} MB in memory')

u, n = np.unique(Y, return_counts=True)
names = dict(enumerate(C.CLASS_NAMES)); names[C.IGNORE_INDEX] = 'no data'
for k, c in zip(u, n):
    print(f'   {names[int(k)]:11s} {c/n.sum():6.1%}')

## 5. Split by place, not at random

Neighbouring patches look alike. Split them at random and the validation
set is full of near-copies of training patches, so the score measures
memory, not generalisation. Here whole blocks of 4 x 4 patches (about
30 km) go to one side or the other.

In [ ]:
blocks = [(gx // 4, gy // 4) for gx, gy in places]
uniq = sorted(set(blocks))
np.random.default_rng(SEED).shuffle(uniq)
fold = {b: i % 5 for i, b in enumerate(uniq)}
is_val = np.array([fold[b] == 0 for b in blocks])
if is_val.all() or not is_val.any():          # too few blocks to split by place
    is_val = np.arange(len(files)) % 5 == 0
    print('!! too few distinct blocks - fell back to a non-spatial split')
print(f'train {(~is_val).sum()} patches, validation {is_val.sum()} patches, '
      f'{len(uniq)} blocks')

## 6. Training loop, shared by both models

Class weights counter imbalance: without them a model can score well by
answering the commonest class everywhere. The score reported is IoU per
class - the overlap of predicted and labelled area, divided by their
union - computed on the validation patches only.

In [ ]:
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F

class Patches(Dataset):
    def __init__(self, X, Y, augment):
        self.X, self.Y, self.augment = X, Y, augment
    def __len__(self):
        return len(self.X)
    def __getitem__(self, i):
        x, y = self.X[i], self.Y[i]
        if self.augment:                       # the ground has no 'up'
            k = np.random.randint(4)
            x, y = np.rot90(x, k, (1, 2)), np.rot90(y, k)
            if np.random.rand() < 0.5:
                x, y = x[:, :, ::-1], y[:, ::-1]
        return (torch.from_numpy(np.ascontiguousarray(x)),
                torch.from_numpy(np.ascontiguousarray(y)).long())

train_ds = Patches(X[~is_val], Y[~is_val], augment=True)
val_ds = Patches(X[is_val], Y[is_val], augment=False)

K = len(C.CLASS_NAMES)
freq = np.bincount(Y[~is_val][Y[~is_val] != C.IGNORE_INDEX], minlength=K) + 1
weights = torch.tensor((freq.sum() / freq) ** 0.5, dtype=torch.float32)
weights = (weights / weights.mean()).to(device)

def logits_of(out):
    return out.output if hasattr(out, 'output') else out   # TerraTorch wraps it

def confusion(model, ds, batch):
    cm = torch.zeros(K, K, dtype=torch.long)
    model.eval()
    with torch.no_grad():
        for x, y in DataLoader(ds, batch_size=batch):
            p = logits_of(model(x.to(device))).argmax(1).cpu()
            ok = y != C.IGNORE_INDEX
            cm += torch.bincount(y[ok] * K + p[ok], minlength=K * K).view(K, K)
    return cm

def iou(cm):
    tp = cm.diag().float()
    return (tp / (cm.sum(0) + cm.sum(1) - tp).clamp(min=1)).numpy()

def train(model, name, epochs, batch, lr):
    model.to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, max(epochs, 1))
    amp = device == 'cuda'
    scaler = torch.amp.GradScaler('cuda', enabled=amp)
    dl = DataLoader(train_ds, batch_size=batch, shuffle=True, drop_last=len(train_ds) > batch)
    best, best_state, history = -1.0, None, []
    for ep in range(1, epochs + 1):
        model.train(); tot = 0.0
        for x, y in dl:
            x, y = x.to(device), y.to(device)
            with torch.autocast(device, dtype=torch.float16, enabled=amp):
                loss = F.cross_entropy(logits_of(model(x)).float(), y, weight=weights,
                                       ignore_index=C.IGNORE_INDEX)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            tot += loss.item() * len(x)
        sched.step()
        scores = iou(confusion(model, val_ds, batch))
        history.append({'epoch': ep, 'loss': tot / len(train_ds),
                        **dict(zip(C.CLASS_NAMES, scores.round(3).tolist()))})
        print(f'{name} epoch {ep:2d}  loss {tot/len(train_ds):.3f}  IoU ' +
              '  '.join(f'{n[:5]} {s:.2f}' for n, s in zip(C.CLASS_NAMES, scores)) +
              f'  | mean {scores.mean():.3f}')
        if scores.mean() > best:
            best = scores.mean()
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    print(f'best mean IoU {best:.3f}')
    if SAVE_WEIGHTS:                          # half precision: half the size
        path = OUT_DIR / f'{name}.pt'
        torch.save({k: v.half() if v.is_floating_point() else v
                    for k, v in best_state.items()}, path)
        print(f'weights saved: {path} ({path.stat().st_size/1e6:.0f} MB)')
    return history

## 7. U-Net

The baseline model: an encoder pre-trained on ordinary photographs
(ImageNet) with its first layer widened to six bands, and a decoder that
turns its features back into a full-size mask. About 24 million parameters;
a few minutes on a T4.

In [ ]:
import segmentation_models_pytorch as smp
unet = smp.Unet('resnet34', encoder_weights='imagenet', in_channels=6, classes=K)
hist_unet = train(unet, 'unet_weak', EPOCHS_UNET, BATCH_UNET, lr=1e-3)

## 8. Prithvi-EO-2.0-300M-TL

Pre-trained on HLS - the same six bands at the same scale - so it starts
from features of satellite imagery rather than photographs. Built exactly
as in the official BurnScars example: 4 of the 24 transformer layers feed
a U-Net-style decoder. One date per patch here; the stacks of dates that
need the DGX Spark come later. About ten times slower than the U-Net.

In [ ]:
hist_prithvi = None
if USE_PRITHVI:
    from terratorch.models import EncoderDecoderFactory
    unet.cpu()
    if device == 'cuda':
        torch.cuda.empty_cache()
    prithvi = EncoderDecoderFactory().build_model(
        task='segmentation',
        backbone='prithvi_eo_v2_300_tl',
        backbone_pretrained=True,
        backbone_bands=['BLUE', 'GREEN', 'RED', 'NIR_NARROW', 'SWIR_1', 'SWIR_2'],
        backbone_img_size=256,
        necks=[{'name': 'SelectIndices', 'indices': [5, 11, 17, 23]},
               {'name': 'ReshapeTokensToImage'},
               {'name': 'LearnedInterpolateToPyramidal'}],
        decoder='UNetDecoder',
        decoder_channels=[512, 256, 128, 64],
        num_classes=K)
    hist_prithvi = train(prithvi, 'prithvi300tl_weak', EPOCHS_PRITHVI, BATCH_PRITHVI, lr=5e-5)
    if device == 'cuda':
        print(f'peak GPU memory: {torch.cuda.max_memory_allocated()/1e9:.1f} GB')

## 9. How closely each model reproduces the threshold

Agreement with the automatic labels, per class. **This is not accuracy.**
A perfect score means a perfect copy of the threshold, which is exactly
what was asked for and exactly what is not the goal. Expect water to score
high and salt versus bare soil to score lower: their boundary in the
labels is an arbitrary line, which no model can learn to place for a
reason.

In [ ]:
import pandas as pd, json
rows = {'U-Net': hist_unet[-1]}
if hist_prithvi:
    rows['Prithvi 300M-TL'] = hist_prithvi[-1]
table = pd.DataFrame(rows).T[list(C.CLASS_NAMES)]
table['mean'] = table.mean(axis=1)
(OUT_DIR / 'results_weak.json').write_text(json.dumps(
    {'unet': hist_unet, 'prithvi': hist_prithvi}, indent=1))
table.round(3)

## 10. Look at the masks

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

# grey for no data: shown as a masked array it would be the figure's white,
# which is indistinguishable from salt
cmap = ListedColormap([(0.12, 0.37, 0.71), (0.95, 0.94, 0.90),
                       (0.65, 0.49, 0.32), (0.24, 0.57, 0.27), (0.45, 0.45, 0.45)])
models = [('U-Net', unet)] + ([('Prithvi', prithvi)] if USE_PRITHVI else [])
idx = np.where(is_val)[0][:4]
mean = np.asarray(C.MODEL_MEAN)[:, None, None]; std = np.asarray(C.MODEL_STD)[:, None, None]

fig, ax = plt.subplots(len(idx), 2 + len(models), figsize=(3.2 * (2 + len(models)), 3.2 * len(idx)),
                       squeeze=False)
for r, i in enumerate(idx):
    refl = (X[i] * std + mean) / C.STORE_SCALE
    rgb = np.clip(np.stack([refl[4], refl[3], refl[2]], -1) / 0.45, 0, 1)   # SWIR1/NIR/red
    panels = [(rgb, 'image'), (np.where(Y[i] == C.IGNORE_INDEX, K, Y[i]), 'threshold')]
    x = torch.from_numpy(X[i:i + 1])
    for n, m in models:
        m.to(device).eval()
        with torch.no_grad():
            panels.append((logits_of(m(x.to(device))).argmax(1)[0].cpu().numpy(), n))
        m.cpu()
    for c, (img, title) in enumerate(panels):
        a = ax[r, c]
        a.imshow(img) if c == 0 else a.imshow(img, cmap=cmap, vmin=0, vmax=K)
        a.set_title(title if r == 0 else ''); a.axis('off')
plt.suptitle('blue water  |  white salt  |  brown bare soil  |  green vegetation  |  grey no data')
plt.tight_layout(); plt.show()

## What comes next

1. Label 100-300 patches by hand in QGIS - the ones `aral_colab.ipynb`
   chose, on boundaries.
2. Load those labels instead of calling `labels.weak_labels`, and keep a
   held-out set the models never train on.
3. Run this notebook again. Now the table answers the real question: on
   ground truth, does either model beat the threshold - and on which class?